In [24]:
import os
import requests
import json
import base64
import zlib
from dotenv import load_dotenv
import unstructured_client
from unstructured_client.models import shared
from unstructured_client.models.errors import SDKError





In [14]:
load_dotenv()

client = unstructured_client.UnstructuredClient(
    api_key_auth=os.getenv("UNSTRUCTURED_API_KEY")
)

PDF_PATH = r"C:\Users\Muneeb Syed\Desktop\RAG\chunking\document.pdf"

In [15]:
def decode_orig_elements(orig_elements_b64: str) -> list[dict]:
    """Reverse the gzip+base64 packing Unstructured applies to orig_elements."""
    decoded = base64.b64decode(orig_elements_b64)
    decompressed = zlib.decompress(decoded)
    return json.loads(decompressed)

In [ ]:
with open(PDF_PATH, "rb") as f:
    req = {
        "partition_parameters": {
            "files": {
                "content": f.read(),
                "file_name": os.path.basename(PDF_PATH),
            },
            "strategy": "hi_res",
            "chunking_strategy": "by_title",
            "max_characters": 1500,
            "new_after_n_chars": 1000,
            "combine_under_n_chars": 500,
            "split_pdf_page": True,
            "split_pdf_allow_failed": True,
            "split_pdf_concurrency_level": 15,

            # image extraction
            "extract_image_block_types": ["Image", "Table"],  # add Table if you want table images too
            "extract_image_block_to_payload": True,

            # make sure orig_elements is included (defaults to True, but be explicit)
            "include_orig_elements": True,
        }
    }

    try:
        res = client.general.partition(request=req)
        raw_chunks = res.elements
        print(f"Got {len(raw_chunks)} chunks")

        enriched_chunks = []
        for chunk in raw_chunks:
            orig_b64 = chunk.get("metadata", {}).get("orig_elements")
            images = []

            if orig_b64:
                orig_elements = decode_orig_elements(orig_b64)
                for el in orig_elements:
                    el_type = el.get("type")
                    img_b64 = el.get("metadata", {}).get("image_base64")
                    if el_type in ("Image", "Table") and img_b64:
                        images.append({
                            "type": el_type,
                            "image_base64": img_b64,
                            "image_mime_type": el.get("metadata", {}).get("image_mime_type"),
                            "page_number": el.get("metadata", {}).get("page_number"),
                        })

            enriched_chunks.append({
                "element_id": chunk["element_id"],
                "text": chunk["text"],
                "page_number": chunk.get("metadata", {}).get("page_number"),
                "images": images,   # <-- your image data lives here now
            })

        with open("document_chunks.json", "w", encoding="utf-8") as out:
            json.dump(raw_chunks, out, indent=2)  # raw, if you still want it

        with open("document_chunks_with_images.json", "w", encoding="utf-8") as out:
            json.dump(enriched_chunks, out, indent=2)

        n_with_images = sum(1 for c in enriched_chunks if c["images"])
        print(f"{n_with_images} chunks contain image data")

    except SDKError as e:
        print("SDK error:", e.message)

INFO: split_pdf event=plan_created operation_id=cb3014c1-3225-4549-9e34-123667735417 filename=document.pdf strategy=hi_res page_range=1-13 page_count=13 split_size=2 chunk_count=7 concurrency=15 allow_failed=True cache_mode=disabled timeout_seconds=None retry_config_mode=sdk_default_or_unset pool_max_connections=100 pool_max_keepalive=20 pool_keepalive_expiry=5.0s tls=trust_store=custom-ca-bundle mtls_cert=none
INFO: HTTP Request: GET https://api.unstructuredapp.io/general/docs "HTTP/1.1 200 OK"
INFO: split_pdf event=batch_start operation_id=cb3014c1-3225-4549-9e34-123667735417 chunk_count=7 concurrency=15 allow_failed=True client_timeout_seconds=None future_timeout_seconds=3605 num_waves=1
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: P

Got 37 chunks
8 chunks contain image data


In [ ]:
#
# this m the format of document_chunk_with images.json
# it is a list with many chunks
#{
#"element_id": str
#"text": str
#"page_number": int
#"images": list  
#}
# where images is an array which would be empty if there is no images and if there are images then it would have elements looking like this 
# 
# 
# {
# "type": 
# "image_base64":
# "image_mime_type":
# "page_number":
# }


                        


In [ ]:
def jina_embedding_model(input_data) :
    url = "https://api.jina.ai/v1/embeddings"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {os.getenv("JINA_API_KEY")}"
    }
    data = {
        "model": "jina-embeddings-v2-base-en",
        "task": "separation",
        "input": input_data
    }
    response = requests.post(url, headers=headers, json=data)
    response.raise_for_status()
    return response

In [28]:
load_dotenv()

from pinecone import Pinecone, ServerlessSpec
EMBED_DIM = 768
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
INDEX_NAME = "rag-with-images"

In [29]:
# Create the index if it doesn't already exist
if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBED_DIM,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(INDEX_NAME)

INFO: Describing index 'rag-with-images'
INFO: HTTP Request: GET https://api.pinecone.io/indexes/rag-with-images "HTTP/1.1 404 Not Found"
INFO: Creating index 'rag-with-images'
INFO: Describing index 'rag-with-images'
INFO: HTTP Request: GET https://api.pinecone.io/indexes/rag-with-images "HTTP/1.1 200 OK"
INFO: Describing index 'rag-with-images'
INFO: HTTP Request: GET https://api.pinecone.io/indexes/rag-with-images "HTTP/1.1 200 OK"
INFO: Index client created for host https://rag-with-images-n86mliq.svc.aped-4627-b74a.pinecone.io


In [58]:
JSON_FILE = "document_chunks_with_images.json"


with open(JSON_FILE, "r", encoding="utf-8") as f:
    chunks = json.load(f)

chunk_texts = [chunk["text"] for chunk in chunks]
chunk_texts
chunk_page = [chunk["page_number"] for chunk in chunks]
chunk_page
chunk_images = [
    [img["image_base64"] for img in chunk["images"]] if chunk["images"] else []
    for chunk in chunks
]
chunk_images[35]


['...']

In [65]:
from io import BytesIO
from PIL import Image

def compress_image_b64_to_target(b64_string, target_kb=10, max_size=(512, 512), start_quality=85):
    if not b64_string:
        return b64_string

    if "," in b64_string:
        b64_string = b64_string.split(",", 1)[1]

    image_bytes = base64.b64decode(b64_string)
    img = Image.open(BytesIO(image_bytes)).convert("RGB")

    img.thumbnail(max_size, Image.Resampling.LANCZOS)

    quality = start_quality
    best_bytes = None

    while quality >= 10:
        out = BytesIO()
        img.save(out, format="JPEG", quality=quality, optimize=True)
        data = out.getvalue()

        best_bytes = data
        if len(data) <= target_kb * 1024:
            break

        quality -= 5

    return base64.b64encode(best_bytes).decode("utf-8")

In [67]:
chunk_images = [
    [compress_image_b64_to_target(img["image_base64"]) for img in chunk["images"]] if chunk["images"] else []
    for chunk in chunks
]
chunk_images[35]

['...']

In [56]:
# ------------------------------------------------------------------
# 3. Prepare and embed your documents
# ------------------------------------------------------------------



result = jina_embedding_model(chunk_texts)

data = sorted(result.json()["data"], key=lambda x: x["index"])
embeddings = [item["embedding"] for item in data]

In [68]:
# ------------------------------------------------------------------
# 4. Upsert (insert/update) vectors into Pinecone
# ------------------------------------------------------------------
vectors_to_upsert = [
    {
        "id": f"doc_{i}",
        "values": embeddings[i],
        "metadata": {"text": chunk_texts[i],
                     "page_number" : chunk_page[i],
                     "image":chunk_images[i]}  # store original text as metadata
    }
    for i in range(len(chunk_texts))
]

index.upsert(vectors=vectors_to_upsert, namespace="my-namespace")

INFO: Upserting 37 vectors into namespace 'my-namespace'


UpsertResponse(upserted_count=37)